In [1]:
!pip install datasets pandas

In [2]:
from datasets import load_dataset

# Load DS2 (contains multiple sources including FPB)
ds2 = load_dataset("sjyuxyz/financial-sentiment-analysis")
print("DS2 loaded:", ds2)

# Load FPB (100% annotator agreement, highest quality)
fpb = load_dataset("FinanceMTEB/financial_phrasebank")
print("FPB loaded:", fpb)
print("FPB sample:", fpb["train"][0])

DS2 loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'source'],
        num_rows: 80029
    })
    test: Dataset({
        features: ['text', 'label', 'source'],
        num_rows: 10004
    })
    valid: Dataset({
        features: ['text', 'label', 'source'],
        num_rows: 10004
    })
})
FPB loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1264
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1000
    })
})
FPB sample: {'text': 'The Samsung Mobile Applications Store was launched in January 2009 by Samsung Mobile Innovator , a program which enables mobile software developers to create applications for use across Samsung mobile devices .', 'label_text': 'neutral', 'label': 1}


In [3]:
# Check FPB field names
print("FPB features:", fpb["train"].features)
print("FPB sample:", fpb["train"][0])

FPB features: {'text': Value('string'), 'label_text': Value('string'), 'label': Value('int64')}
FPB sample: {'text': 'The Samsung Mobile Applications Store was launched in January 2009 by Samsung Mobile Innovator , a program which enables mobile software developers to create applications for use across Samsung mobile devices .', 'label_text': 'neutral', 'label': 1}


In [4]:
# Check DS2 label mapping
print("=== DS2 Label Check ===")
for label_id in [0, 1, 2]:
    examples = [x for x in ds2["train"] if x["label"] == label_id][:3]
    print(f"\nlabel={label_id}, examples:")
    for ex in examples:
        print(f"  [{ex['source']}] {ex['text'][:80]}")

# Check FPB label mapping (field name is "text" and "label_text", not "sentence"!)
print("\n=== FPB Label Check ===")
for label_id in [0, 1, 2]:
    examples = [x for x in fpb["train"] if x["label"] == label_id][:3]
    print(f"\nlabel={label_id}, examples:")
    for ex in examples:
        print(f"  label_text={ex['label_text']}  |  {ex['text'][:80]}")

=== DS2 Label Check ===

label=0, examples:
  [TimKoornstra/financial-tweets-sentiment] LAS VEGAS, NV / ACCESSWIRE / March 23, 2021 / Winners, Inc. (OTC PINK:WNRS) subs
  [TimKoornstra/financial-tweets-sentiment] Aristocrat Leisure Limited (ASX:ALL) Earns A Nice Return On Capital Employed
  [zeroshot/twitter-financial-news-sentiment] Lenders, Homebuyers Approve NBCC’s Bid For Jaypee Infra

label=1, examples:
  [FinGPT/fingpt-sentiment-cls] Fed's Rosengren says he is 'optimistic' on economy
  [FinGPT/fingpt-sentiment-cls] Silver Price Forecast – Silver Markets Continue Forming Support
  [TimKoornstra/financial-tweets-sentiment] $ENPH - Enphase: Promising Growth Vision. https://t.co/LoJocgYZfq #investing #fi

label=2, examples:
  [TimKoornstra/financial-tweets-sentiment] $AAPL weekly still under the 50 moving average and creating a lower high. https:
  [TimKoornstra/financial-tweets-sentiment] Mass layoffs push Canada consumer confidence to all-time low https://t.co/lj9u4j
  [FinGPT/fing

In [5]:
from datasets import load_dataset

ds = load_dataset("TheFinAI/flare-sm-acl")

In [6]:
# Check FLARE structure
print(ds)

print("\nFeatures:", ds["train"].features)

print("\nSample:", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
        num_rows: 20781
    })
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
        num_rows: 3720
    })
    valid: Dataset({
        features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
        num_rows: 2555
    })
})

Features: {'id': Value('string'), 'query': Value('string'), 'answer': Value('string'), 'text': Value('string'), 'choices': List(Value('string')), 'gold': Value('int64')}

Sample: {'id': 'aclsm0', 'query': 'Scrutinize the data and tweets to envisage if the closing price of $csco will swell or contract at 2014-01-02. Please affirm either Rise or Fall.\nContext: date,open,high,low,close,adj-close,inc-5,inc-10,inc-15,inc-20,inc-25,inc-30\n2013-12-17,-0.9,1.0,-1.1,1.2,1.2,-1.3,-0.0,0.4,0.9,2.1,3.6\n2013-12-18,-0.6,0.2,-2.3,0.4,0.4,-1.6,-0.5,-0.0,0.4,1.2,2.9\n2013-12-19,0.0,0.0,-1.4,0.3,0.3,-1.4,-0.8,-0.4,0.0,0.

In [7]:
import json

INSTRUCTION = (
    "You are a financial analyst. "
    "Analyze the sentiment of the following financial text. "
    "Answer with positive or negative."
)

# -----------------------
# DS2
# -----------------------
LABEL_MAP_DS2 = {
    0: "neutral",
    1: "positive",
    2: "negative"
}

def format_ds2(example):
    label = LABEL_MAP_DS2[example["label"]]
    
    if label == "neutral":
        return None  # ❗过滤掉
    
    return {
        "instruction": INSTRUCTION,
        "input": example["text"],
        "output": label,
        "source": example["source"]
    }

In [8]:
LABEL_MAP_FPB = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

def format_fpb(example):
    label = example["label"]
    
    if isinstance(label, int):
        label = LABEL_MAP_FPB[label]
    
    if label == "neutral":
        return None  # ❗过滤
    
    return {
        "instruction": INSTRUCTION,
        "input": example["text"],
        "output": label,
        "source": "financial_phrasebank"
    }

In [9]:
def format_flare(example):
    label = example["gold"]
    
    # 0 = Rise, 1 = Fall
    label_str = "positive" if label == 0 else "negative"
    
    return {
        "instruction": INSTRUCTION,
        "input": example["query"],  # ⚠️ 用 query！
        "output": label_str,
        "source": "flare-sm-acl"
    }

In [10]:
# DS2
ds2_train = [x for x in (format_ds2(e) for e in ds2["train"]) if x]
ds2_test  = [x for x in (format_ds2(e) for e in ds2["test"]) if x]

# FPB
fpb_all   = [x for x in (format_fpb(e) for e in fpb["train"]) if x]
split_idx = int(len(fpb_all) * 0.8)
fpb_train = fpb_all[:split_idx]
fpb_test  = fpb_all[split_idx:]

# FLARE
flare_train = [format_flare(x) for x in ds["train"]]
flare_test  = [format_flare(x) for x in ds["test"]]

In [11]:
# DS2
ds2_train = [x for x in (format_ds2(e) for e in ds2["train"]) if x]
ds2_test  = [x for x in (format_ds2(e) for e in ds2["test"]) if x]

# FPB
fpb_all   = [x for x in (format_fpb(e) for e in fpb["train"]) if x]
split_idx = int(len(fpb_all) * 0.8)
fpb_train = fpb_all[:split_idx]
fpb_test  = fpb_all[split_idx:]

# FLARE
flare_train = [format_flare(x) for x in ds["train"]]
flare_test  = [format_flare(x) for x in ds["test"]]

In [12]:
train_data = ds2_train + fpb_train + flare_train
test_data  = ds2_test  + fpb_test  + flare_test

print(f"Total train: {len(train_data)}")
print(f"Total test:  {len(test_data)}")

Total train: 84176
Total test:  11699


In [13]:
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

{
  "instruction": "You are a financial analyst. Analyze the sentiment of the following financial text. Answer with positive or negative.",
  "input": "Fed's Rosengren says he is 'optimistic' on economy",
  "output": "positive",
  "source": "FinGPT/fingpt-sentiment-cls"
}


In [14]:
from collections import Counter
import json
import random

# -------------------------
# Merge ALL datasets
# -------------------------
all_train = ds2_train + fpb_train + flare_train
all_test  = ds2_test  + fpb_test  + flare_test

print(f"Merged train: {len(all_train)} samples")
print(f"Merged test:  {len(all_test)} samples")

# -------------------------
# Shuffle（一定要加！）
# -------------------------
random.shuffle(all_train)
random.shuffle(all_test)

# -------------------------
# Label distribution
# -------------------------
label_counts = Counter(x["output"] for x in all_train)
print(f"\nLabel distribution (train): {dict(label_counts)}")

# -------------------------
# Source distribution（很重要）
# -------------------------
source_counts = Counter(x["source"] for x in all_train)
print(f"\nSource distribution (train): {dict(source_counts)}")

# -------------------------
# Save
# -------------------------
with open("train_financial_sentiment_merged.json", "w", encoding="utf-8") as f:
    json.dump(all_train, f, ensure_ascii=False, indent=2)

with open("test_financial_sentiment_merged.json", "w", encoding="utf-8") as f:
    json.dump(all_test, f, ensure_ascii=False, indent=2)

print("\nDone!")
print("train_financial_sentiment_merged.json  -> training")
print("test_financial_sentiment_merged.json   -> evaluation only")

Merged train: 84176 samples
Merged test:  11699 samples

Label distribution (train): {'negative': 32660, 'positive': 51516}

Source distribution (train): {'flare-sm-acl': 20781, 'FinGPT/fingpt-sentiment-cls': 37968, 'TimKoornstra/financial-tweets-sentiment': 20806, 'zeroshot/twitter-financial-news-sentiment': 2656, 'takala/financial_phrasebank': 1577, 'financial_phrasebank': 388}

Done!
train_financial_sentiment_merged.json  -> training
test_financial_sentiment_merged.json   -> evaluation only


In [15]:
import json

with open("train_financial_sentiment_merged.json", "r", encoding="utf-8") as f:
    train = json.load(f)

with open("test_financial_sentiment_merged.json", "r", encoding="utf-8") as f:
    test = json.load(f)

from collections import Counter
print(f"Train: {len(train)} samples")
print(f"Test:  {len(test)} samples")
print(f"\nTrain label distribution: {dict(Counter(x['output'] for x in train))}")
print(f"Test  label distribution: {dict(Counter(x['output'] for x in test))}")

Train: 84176 samples
Test:  11699 samples

Train label distribution: {'negative': 32660, 'positive': 51516}
Test  label distribution: {'positive': 7143, 'negative': 4556}


*CoT-SFT*

In [16]:
import json
import time
import os
from openai import OpenAI

# ============================================================
# Configuration
# ============================================================

API_KEY = "sk-81e51cd9fd464ee6bc2f950f7d2a0e74"

INPUT_FILE   = "train_text_label.json"
OUTPUT_FILE  = "train_cot_sft.json"
ERROR_FILE   = "train_cot_errors.json"
CHECKPOINT_DIR = "checkpoints"

MAX_SAMPLES    = None   # None = run all
SLEEP_SEC      = 0.1
SAVE_EVERY     = 10000  # save checkpoint every N samples

# ============================================================

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com"
)

INSTRUCTION_COT = (
    "You are a financial analyst. "
    "Analyze the sentiment of the following financial text. "
    "Think step by step before giving your final answer. "
    "Answer with positive, negative, or neutral."
)

# ============================================================

def generate_cot(text: str, label: str) -> str:
    prompt = f"""You are an expert financial analyst.

Financial text: "{text}"

Explain why the sentiment is {label}.

Format EXACTLY like this:

<think>
3-4 sentence reasoning
</think>

{label}
"""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()


def save_checkpoint(results: list, errors: list, checkpoint_idx: int):
    """Save a numbered checkpoint file every SAVE_EVERY samples."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{checkpoint_idx:07d}.json")
    with open(ckpt_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    # Always overwrite errors file with latest
    with open(ERROR_FILE, "w", encoding="utf-8") as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)
    print(f"  → Checkpoint saved: {ckpt_path}  ({len(results)} results so far)")


def load_existing_progress():
    """
    Resume from the latest checkpoint if OUTPUT_FILE already exists,
    or scan checkpoint dir for the highest checkpoint.
    Returns (results_so_far, errors_so_far, start_index).
    """
    # Check final output first
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            results = json.load(f)
        errors = []
        if os.path.exists(ERROR_FILE):
            with open(ERROR_FILE, "r", encoding="utf-8") as f:
                errors = json.load(f)
        start = len(results) + len(errors)
        print(f"Resuming from final output file: {start} items already done.")
        return results, errors, start

    # Scan checkpoints
    if os.path.isdir(CHECKPOINT_DIR):
        ckpts = sorted([
            f for f in os.listdir(CHECKPOINT_DIR)
            if f.startswith("checkpoint_") and f.endswith(".json")
        ])
        if ckpts:
            latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
            with open(latest, "r", encoding="utf-8") as f:
                results = json.load(f)
            errors = []
            if os.path.exists(ERROR_FILE):
                with open(ERROR_FILE, "r", encoding="utf-8") as f:
                    errors = json.load(f)
            start = len(results) + len(errors)
            print(f"Resuming from checkpoint '{ckpts[-1]}': {start} items already done.")
            return results, errors, start

    return [], [], 0

# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    train_data = json.load(f)

if MAX_SAMPLES:
    train_data = train_data[:MAX_SAMPLES]

total = len(train_data)
print(f"Total samples in file: {total}")

# ---- Resume support ----
results, errors, start_index = load_existing_progress()
print(f"Starting from index {start_index}, {total - start_index} samples remaining.\n")

# ---- Main loop ----
for i, example in enumerate(train_data[start_index:], start=start_index):

    text  = example["input"]
    label = example["output"]

    try:
        cot = generate_cot(text, label)
        results.append({
            "instruction": INSTRUCTION_COT,
            "input":       text,
            "output":      cot,
            "source":      example.get("source", "unknown")
        })
        print(f"[{i+1}/{total}] ✓")

    except Exception as e:
        print(f"[{i+1}/{total}] ERROR: {e}")
        errors.append({"index": i, "text": text, "error": str(e)})

    # ---- Checkpoint every SAVE_EVERY processed samples ----
    processed = (i + 1) - start_index          # how many we did this run
    total_done = len(results) + len(errors)     # absolute count

    if processed % SAVE_EVERY == 0:
        save_checkpoint(results, errors, total_done)

    time.sleep(SLEEP_SEC)

# ============================================================
# Final save
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open(ERROR_FILE, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\n✅ Done! {len(results)} succeeded, {len(errors)} errors.")
print(f"Output → {OUTPUT_FILE}")
if results:
    print("\nFirst result preview:")
    print(json.dumps(results[0], indent=2, ensure_ascii=False))

Total samples in file: 84176
Resuming from checkpoint 'checkpoint_0060000.json': 60000 items already done.
Starting from index 60000, 24176 samples remaining.

[60001/84176] ✓
[60002/84176] ✓
[60003/84176] ✓
[60004/84176] ✓
[60005/84176] ✓
[60006/84176] ✓
[60007/84176] ✓
[60008/84176] ✓
[60009/84176] ✓
[60010/84176] ✓
[60011/84176] ✓
[60012/84176] ✓
[60013/84176] ✓
[60014/84176] ✓
[60015/84176] ✓
[60016/84176] ✓
[60017/84176] ✓
[60018/84176] ✓
[60019/84176] ✓
[60020/84176] ✓
[60021/84176] ✓
[60022/84176] ✓
[60023/84176] ✓
[60024/84176] ✓
[60025/84176] ✓
[60026/84176] ✓
[60027/84176] ✓
[60028/84176] ✓
[60029/84176] ✓
[60030/84176] ✓
[60031/84176] ✓
[60032/84176] ✓
[60033/84176] ✓
[60034/84176] ✓
[60035/84176] ✓
[60036/84176] ✓
[60037/84176] ✓
[60038/84176] ✓
[60039/84176] ✓
[60040/84176] ✓
[60041/84176] ✓
[60042/84176] ✓
[60043/84176] ✓
[60044/84176] ✓
[60045/84176] ✓
[60046/84176] ✓
[60047/84176] ✓
[60048/84176] ✓
[60049/84176] ✓
[60050/84176] ✓
[60051/84176] ✓
[60052/84176] ✓
[60053/8